# 01 — Bronze: descarga

**Fase 1 · Capa Bronze** · interpretación en `data/bronze/bronze.md`

Lee `config.yaml → fuentes` y deja cada archivo en una ruta fija: `data/bronze/<fuente>/[<edición>/]<archivo>` (D-014).

- **Sin cambios en el servidor** (mismo tamaño y `Last-Modified`, o mismo sha256): no se toca nada.
- **Versión nueva**: se descarga a `<archivo>.part`, se valida y recién entonces **sobrescribe** la anterior. Si la descarga falla, la versión anterior queda intacta.
- `manifest.json` es el inventario de lo que hay hoy en Bronze (una entrada por archivo vigente, con `fecha_descarga` y `run_id` de la sesión que lo bajó).
- La bitácora guarda, por fuente, archivos, filas, tamaño y fecha de los datos; de ahí sale `data/auditoria/volumenes.parquet`.

| Fuente | Qué se descarga |
| --- | --- |
| F1 SIAF | 2009–2024 anual y 2025–2026 *Diario* (el *Mensual* se descarta) |
| F2 SISMEPRE | 7 CSV (se publican completos, no por año) |
| F3 RENAMU | ediciones 2009–2026; los ZIP se guardan y se descomprimen junto al ZIP (el análisis usa 2020–2026) |


## 1. Configuración

In [1]:
import sys
from pathlib import Path

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ARQUITECTURA.md").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import uuid
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm

from src.config import cargar_config, cargar_fuentes, cargar_reglas, ruta
from src import bronze, auditoria

cfg = cargar_config()
FECHA = bronze.fecha_local()     # fecha de la sesión (hora de Lima)
INICIO_UTC = bronze._ahora()
RUN_ID = f"bronze_descarga_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:6]}"
FORZAR = False        # True: descargar y sobrescribir aunque el servidor no haya cambiado el archivo

print("Fecha de la sesión:", FECHA)
print("Id de ejecución   :", RUN_ID)

Fecha de la sesión: 2026-11-22
Id de ejecución   : bronze_descarga_20261122T173134Z_871372


/home/rodri/proyectos/Final_BI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Catálogo de recursos

In [2]:
cargar_config.cache_clear()
recursos = bronze.construir_catalogo(cargar_fuentes())

df_catalogo = pd.DataFrame([{
    "fuente": r.fuente, "anio": r.anio, "archivo": r.archivo, "modo": r.modo,
    "descomprimir": r.descomprimir, "destino": str(r.destino().relative_to(RAIZ)), "url": r.url,
} for r in recursos])

print(df_catalogo.groupby("fuente").size().rename("recursos").to_string())
df_catalogo

fuente
F1    18
F2     7
F3    19


,fuente,anio,archivo,modo,descomprimir,destino,url
0,F1,2009.0,2009-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2009-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
1,F1,2010.0,2010-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2010-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
2,F1,2011.0,2011-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2011-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
3,F1,2012.0,2012-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2012-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
4,F1,2013.0,2013-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2013-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
5,F1,2014.0,2014-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2014-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
6,F1,2015.0,2015-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2015-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
7,F1,2016.0,2016-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2016-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
8,F1,2017.0,2017-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2017-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
9,F1,2018.0,2018-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2018-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...


## 3. Descarga

Estados: `descargado` (no estaba en Bronze) · `actualizado` (versión nueva, sobrescribe la anterior) · `sin_cambios` · `error` (la versión anterior queda intacta). El manifest se guarda después de cada recurso que lo cambie.

In [3]:
cargar_config.cache_clear()
manifest = bronze.leer_manifest()
resultados = []
http = bronze.sesion_http()

for r in tqdm(recursos, desc="Descargando"):
    try:
        res = bronze.procesar_recurso(r, manifest, http, forzar=FORZAR)
    except Exception as e:
        res = bronze.Resultado(r, "error", str(e))
    if bronze.registrar_resultado(manifest, res, RUN_ID):
        bronze.guardar_manifest(manifest)
    resultados.append(res)
    tqdm.write(f"{res.estado:<12} {r.fuente} {r.archivo:<40} {res.detalle}")

df_res = pd.DataFrame([{
    "fuente": x.recurso.fuente, "archivo": x.recurso.archivo, "estado": x.estado, "detalle": x.detalle,
    "archivos_registrados": len(x.entradas), "MB_descargados": round(x.bytes_descargados / 1e6, 1),
    "segundos": round(x.segundos, 1),
} for x in resultados])
print()
print(df_res.groupby(["fuente", "estado"]).size().rename("recursos").to_string())

Descargando:  11%|███████▉                                                              | 5/44 [00:00<00:01, 22.17it/s]

sin_cambios  F1 2009-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2010-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2011-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2012-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2013-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2014-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2015-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2016-Ingreso.csv                         vigente desde 2026-11-20


Descargando:  30%|████████████████████▍                                                | 13/44 [00:00<00:01, 30.32it/s]

sin_cambios  F1 2017-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2018-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2019-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2020-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2021-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2022-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2023-Ingreso.csv                         vigente desde 2026-11-20


Descargando:  48%|████████████████████████████████▉                                    | 21/44 [00:00<00:00, 31.14it/s]

sin_cambios  F1 2024-Ingreso.csv                         vigente desde 2026-11-20
sin_cambios  F1 2025-Ingreso-Diario.csv                  vigente desde 2026-11-22
sin_cambios  F1 2026-Ingreso-Diario.csv                  vigente desde 2026-11-22
sin_cambios  F2 rentas_preguntas.csv                     vigente desde 2026-11-22
sin_cambios  F2 rentas_estadistica.csv                   vigente desde 2026-11-22
sin_cambios  F2 rentas_formulario.csv                    vigente desde 2026-11-22
sin_cambios  F2 rentas_esat_estadistica_atm.csv          vigente desde 2026-11-22


... (10 outputs left out)

Descargando:  93%|████████████████████████████████████████████████████████████████▎    | 41/44 [00:02<00:00, 13.84it/s]

sin_cambios  F3 2022.zip                                 vigente desde 2026-11-20
sin_cambios  F3 2023.zip                                 vigente desde 2026-11-20
sin_cambios  F3 2024.zip                                 vigente desde 2026-11-20


Descargando: 100%|█████████████████████████████████████████████████████████████████████| 44/44 [00:02<00:00, 17.18it/s]

sin_cambios  F3 2025_F01_municipalidades.zip             vigente desde 2026-11-20
sin_cambios  F3 2025_F02_centros_poblados.zip            vigente desde 2026-11-20
sin_cambios  F3 2026.zip                                 vigente desde 2026-11-20

fuente  estado     
F1      sin_cambios    18
F2      sin_cambios     7
F3      sin_cambios    19


## 4. Resumen de Bronze

In [4]:
manifest = bronze.leer_manifest()
df_m = pd.DataFrame(manifest["archivos"])
df_m["ext"] = df_m["archivo"].str.rsplit(".", n=1).str[-1].str.lower()

resumen = (df_m.groupby(["fuente", "origen", "ext"])
             .agg(archivos=("ruta", "count"), MB=("bytes", lambda s: round(s.sum() / 1e6, 1)),
                  filas=("filas", lambda s: int(s.fillna(0).sum())), fecha_descarga=("fecha_descarga", "max"))
             .reset_index())
display(resumen)
print(f"Total Bronze: {len(df_m)} archivos · {df_m['bytes'].sum() / 1e9:.2f} GB")

nuevos = df_m[df_m["run_id"] == RUN_ID]
print(f"Archivos escritos en esta sesión: {len(nuevos)} · {nuevos['bytes'].sum() / 1e6:,.1f} MB")

,fuente,origen,ext,archivos,MB,filas,fecha_descarga
0,F1,descarga,csv,18,7336.2,15204055,2026-11-22
1,F2,descarga,csv,7,59.1,493843,2026-11-22
2,F3,descarga,zip,19,64.1,0,2026-11-20
3,F3,extraido_de_zip,csv,1028,271.6,2240471,2026-11-20
4,F3,extraido_de_zip,pdf,62,18.9,0,2026-11-20


Total Bronze: 1134 archivos · 7.75 GB
Archivos escritos en esta sesión: 0 · 0.0 MB


In [5]:
# Detalle de SIAF y SISMEPRE (un archivo = una tabla)
display(df_m[df_m["fuente"].isin(["F1", "F2"])][["fuente", "anio", "archivo", "bytes", "filas", "last_modified", "fecha_descarga", "run_id"]]
        .assign(MB=lambda d: (d["bytes"] / 1e6).round(1)).drop(columns="bytes"))

,fuente,anio,archivo,filas,last_modified,fecha_descarga,run_id,MB
1109,F1,2009.0,2009-Ingreso.csv,1235428.0,"Wed, 21 Oct 2026 03:52:23 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,578.9
1110,F1,2010.0,2010-Ingreso.csv,1320104.0,"Wed, 21 Oct 2026 03:54:18 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,622.4
1111,F1,2011.0,2011-Ingreso.csv,1348362.0,"Wed, 21 Oct 2026 03:57:12 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,637.2
1112,F1,2012.0,2012-Ingreso.csv,759578.0,"Wed, 21 Oct 2026 04:23:47 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,367.0
1113,F1,2013.0,2013-Ingreso.csv,721663.0,"Wed, 21 Oct 2026 20:37:07 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,348.6
1114,F1,2014.0,2014-Ingreso.csv,753998.0,"Wed, 21 Oct 2026 21:21:46 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,364.8
1115,F1,2015.0,2015-Ingreso.csv,781007.0,"Wed, 21 Oct 2026 22:51:01 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,380.7
1116,F1,2016.0,2016-Ingreso.csv,764731.0,"Wed, 21 Oct 2026 23:52:34 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,372.3
1117,F1,2017.0,2017-Ingreso.csv,683816.0,"Thu, 22 Oct 2026 01:34:24 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,332.7
1118,F1,2018.0,2018-Ingreso.csv,656859.0,"Thu, 22 Oct 2026 02:23:29 GMT",2026-11-20,bronze_descarga_20261120T214607Z_ccafe3,319.1


In [6]:
# RENAMU: CSV extraídos por edición
ren = df_m[(df_m["fuente"] == "F3") & (df_m["origen"] == "extraido_de_zip")]
display(ren.groupby("anio").agg(csv=("ext", lambda s: (s == "csv").sum()), pdf=("ext", lambda s: (s == "pdf").sum()),
                                MB=("bytes", lambda s: round(s.sum() / 1e6, 1)), filas_csv=("filas", lambda s: int(s.fillna(0).sum()))))

,csv,pdf,MB,filas_csv
anio,,,,
2009.0,98,3,18.1,211773
2010.0,98,3,17.0,205443
2011.0,98,3,17.6,205180
2012.0,108,4,21.2,220566
2013.0,107,4,20.9,218723
2014.0,96,3,18.0,196541
2015.0,115,6,19.8,213860
2016.0,111,6,18.3,207037
2017.0,116,6,21.9,279535


## 5. Reglas de calidad de Bronze

Se evalúan las reglas `BRZ-TODAS-*` de `config.yaml → reglas_calidad`.

In [7]:
FIRMAS_HTML = bronze.FIRMAS_HTML
reglas = {r["id"]: r for r in cargar_reglas()}
auto = [r for r in recursos if r.modo == "automatica"]
urls_ok = {e["url"] for e in manifest["archivos"] if e["origen"] == "descarga" and (RAIZ / e["ruta"]).exists()}

def _es_html(p):
    with open(p, "rb") as f:
        return f.read(512).lstrip().lower().startswith(FIRMAS_HTML)

evaluaciones = []
def registrar(rid, evaluadas, fallidas, detalle=""):
    r = reglas[rid]
    estado = "OK" if not fallidas else ("FALLA" if r["severidad"] == "critica" else "ALERTA")
    evaluaciones.append({"run_id": RUN_ID, "fecha_utc": bronze._ahora(), "capa": "bronze", "regla": rid,
                         "severidad": r["severidad"], "accion": r["accion"], "filas_evaluadas": evaluadas,
                         "filas_fallidas": len(fallidas), "estado": estado, "detalle": detalle or "; ".join(fallidas[:10])})

# 001: cada recurso automático tiene su descarga registrada y en disco
registrar("BRZ-TODAS-001", len(auto), [r.archivo for r in auto if r.url not in urls_ok])

# 002: sha256 del disco == manifest (integridad). Se recalcula para todo Bronze.
fallas_hash = []
for e in tqdm(manifest["archivos"], desc="sha256"):
    p = RAIZ / e["ruta"]
    if not p.exists() or bronze.sha256_archivo(p) != e["sha256"]:
        fallas_hash.append(e["ruta"])
registrar("BRZ-TODAS-002", len(manifest["archivos"]), fallas_hash)

# 003: ningún archivo vacío ni HTML
registrar("BRZ-TODAS-003", len(manifest["archivos"]),
          [e["ruta"] for e in manifest["archivos"] if e["bytes"] == 0 or _es_html(RAIZ / e["ruta"])])

df_cal = pd.DataFrame(evaluaciones)
display(df_cal[["regla", "severidad", "filas_evaluadas", "filas_fallidas", "estado", "detalle"]])

sha256: 100%|█████████████████████████████████████████████████████████████████████| 1134/1134 [00:08<00:00, 140.44it/s]


,regla,severidad,filas_evaluadas,filas_fallidas,estado,detalle
0,BRZ-TODAS-001,critica,44,0,OK,
1,BRZ-TODAS-002,critica,1134,0,OK,
2,BRZ-TODAS-003,critica,1134,0,OK,


## 6. Bitácora de la ejecución

Una línea en `data/bronze/bitacora_ejecuciones.jsonl`. El bloque `volumenes` (una entrada por fuente: archivos, filas, bytes, fecha de los datos) tiene el mismo formato en todas las capas; `src.auditoria` lo consolida en `data/auditoria/volumenes.parquet`.

In [8]:
volumenes = bronze.volumenes_bronze(manifest, resultados)
bitacora = {
    "run_id": RUN_ID, "notebook": "01_bronze_descarga", "capa": "bronze",
    "fecha": FECHA, "inicio_utc": INICIO_UTC, "fin_utc": bronze._ahora(),
    "recursos": len(recursos), "estados": df_res["estado"].value_counts().to_dict(),
    "volumenes": volumenes,
    "bronze_total_archivos": len(manifest["archivos"]),
    "bronze_total_gb": round(sum(e["bytes"] for e in manifest["archivos"]) / 1e9, 3),
    "calidad": {ev["regla"]: ev["estado"] for ev in evaluaciones},
}
auditoria.escribir_bitacora("bronze", bitacora)

display(pd.DataFrame(volumenes).assign(MB=lambda d: (d["bytes"] / 1e6).round(1),
                                       MB_descargados=lambda d: (d["bytes_descargados"] / 1e6).round(1))
          .drop(columns=["bytes", "bytes_descargados"]))

criticas = df_cal[(df_cal["severidad"] == "critica") & (df_cal["estado"] != "OK")]
assert criticas.empty, f"Reglas críticas fallidas:\n{criticas}"
print("Reglas críticas: OK")

,capa,fuente,tabla,archivos,filas,fecha_datos,recursos,descargados,sin_cambios,errores,MB,MB_descargados
0,bronze,F1,siaf_ingreso,18,15204055,2026-11-22,18,0,18,0,7336.2,0.0
1,bronze,F2,sismepre_predial,7,493843,2026-11-22,7,0,7,0,59.1,0.0
2,bronze,F3,renamu,1109,2240471,2026-11-20,19,0,19,0,354.5,0.0


Reglas críticas: OK


## 7. Auditoría de volúmenes

Se reconstruye `data/auditoria/volumenes.parquet` con todas las bitácoras que traen `volumenes` (una fila por ejecución × capa × fuente). Abajo, la última medición de cada capa y fuente.

In [9]:
df_aud = auditoria.construir_auditoria()
print(f"{len(df_aud)} filas en {ruta('auditoria').relative_to(RAIZ)}/volumenes.parquet")
display(auditoria.ultimo_volumen(df_aud)[["capa", "fuente", "tabla", "archivos", "filas", "mb", "fecha_datos", "fin_utc", "run_id"]])

6 filas en data/auditoria/volumenes.parquet


,capa,fuente,tabla,archivos,filas,mb,fecha_datos,fin_utc,run_id
0,bronze,F1,siaf_ingreso,18,15204055,7336.2,2026-11-22,2026-11-22 17:37:44+00:00,bronze_descarga_20261122T173134Z_871372
1,bronze,F2,sismepre_predial,7,493843,59.1,2026-11-22,2026-11-22 17:37:44+00:00,bronze_descarga_20261122T173134Z_871372
2,bronze,F3,renamu,1109,2240471,354.5,2026-11-20,2026-11-22 17:37:44+00:00,bronze_descarga_20261122T173134Z_871372
